In [1]:
!nvidia-smi

Fri Oct  9 05:26:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from numba import cuda
import os
device = cuda.get_current_device()
print(f"Device Name: {device.name.decode('utf-8')}")
print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {getattr(device, 'MULTIPROCESSOR_COUNT', 'N/A')}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y},
{device.MAX_BLOCK_DIM_Z}")
print(f"Warp Size: {device.WARP_SIZE}")

SyntaxError: unterminated f-string literal (detected at line 8) (3070122824.py, line 8)

In [4]:
from numba import cuda

device = cuda.get_current_device()

name = device.name
print(f"Device Name: {name.decode('utf-8') if isinstance(name, bytes) else name}")

print(f"Compute Capability: {device.compute_capability}")
print(f"Multiprocessor Count (SMs): {device.MULTIPROCESSOR_COUNT}")
print(f"Device ID: {device.id}")
print(f"Warp Size: {device.WARP_SIZE}")
print(f"Max Threads Per Block: {device.MAX_THREADS_PER_BLOCK}")
print(f"Max Block Dimensions: {device.MAX_BLOCK_DIM_X}, {device.MAX_BLOCK_DIM_Y}, {device.MAX_BLOCK_DIM_Z}")

Device Name: Tesla T4
Compute Capability: ComputeCapability(major=7, minor=5)
Multiprocessor Count (SMs): 40
Device ID: 0
Warp Size: 32
Max Threads Per Block: 1024
Max Block Dimensions: 1024, 1024, 64


In [6]:
import numpy as np
import time
from numba import cuda

# 1. CPU Reference Implementation (Pure Python loop)
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# 2. GPU Kernel Implementation
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    # Calculate unique 1D thread ID
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [7]:
def benchmark_run(N, student_id_seed=1234):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)
    # Measure CPU Pure Loop (skip for N >= 10M if too slow)
    if N <= 1_000_000:
        t0 = time.perf_counter()
        _ = double_cpu_pure(h_in)
        cpu_time_ms = (time.perf_counter() - t0) * 1000.0
    else:
        cpu_time_ms = float('nan') # Extrapolate or skip
    # GPU Setup
    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
    # Warmup compilation run
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()
    # GPU Roundtrip Measurement (Includes PCIe transfers)
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0
    # GPU Compute-Only Measurement (Pre-allocated in VRAM)
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0
    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [9]:
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]
print(f"{'N':>12} {'CPU ms':>12} {'GPU total ms':>14} {'GPU kernel ms':>15}")
for N in sizes:
    cpu, tot, ker = benchmark_run(N)
    print(f"{N:>12,} {cpu:>12.3f} {tot:>14.3f} {ker:>15.4f}")

           N       CPU ms   GPU total ms   GPU kernel ms
         100        0.041          2.241          0.1098
       1,000        0.233          0.426          0.0575
      10,000        2.311          0.454          0.0572
     100,000       37.962          0.693          0.0775
   1,000,000      243.360          4.658          0.1101
   5,000,000          nan         12.529          0.2602
  10,000,000          nan         24.392          0.4861


In [10]:
for N in [5_000_000, 10_000_000]:
    np.random.seed(1234)
    h = np.random.rand(N).astype(np.float32)
    t0 = time.perf_counter()
    double_cpu_pure(h)
    print(N, (time.perf_counter() - t0) * 1000.0, "ms")

5000000 1247.0922560000872 ms
10000000 2461.8730360000427 ms


In [11]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    # FORBIDDEN: Writing without bounds check!
    d_arr[idx] = 999.0

# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [12]:
@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)
    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

# Configuration
N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block  # Exactly 5 blocks = 40 threads

log_b = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_t = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_g = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)
log_a = cuda.device_array(blocks_per_grid * threads_per_block, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](log_b, log_t, log_g, log_a, N)
cuda.synchronize()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | {'Status':<10}")
print("-" * 50)
for i in range(32, 40):  # Print the critical boundary transition (Block 4)
    status = "ACTIVE" if log_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{log_g[i]:<10} | {log_b[i]:<10} | {log_t[i]:<10} | {status:<10}")

Global ID  | Block ID   | Thread ID  | Status    
--------------------------------------------------
32         | 4          | 0          | ACTIVE    
33         | 4          | 1          | ACTIVE    
34         | 4          | 2          | ACTIVE    
35         | 4          | 3          | ACTIVE    
36         | 4          | 4          | ACTIVE    
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [13]:
import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
d_arr = cuda.to_device(np.ones(N, dtype=np.float32))
tpb = 256
bpg = (N + tpb - 1) // tpb

# Warm-up so JIT compile time doesn't pollute the timings
d_w = cuda.to_device(np.zeros(1, dtype=np.float32))
naive_sum_kernel[bpg, tpb](d_arr, d_w)
atomic_sum_kernel[bpg, tpb](d_arr, d_w)
cuda.synchronize()

# 5 naive trials (reset the total each time)
print("NAIVE KERNEL")
for trial in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[bpg, tpb](d_arr, d_total)
    cuda.synchronize()
    out = d_total.copy_to_host()[0]
    print(f"Trial {trial}: output = {out}, lost updates = {50000.0 - out}")

# Atomic run
d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
atomic_sum_kernel[bpg, tpb](d_arr, d_total)
cuda.synchronize()
print("ATOMIC output =", d_total.copy_to_host()[0])

# Timing (average of 20 runs each)
def time_kernel(kernel):
    times = []
    for _ in range(20):
        d_t = cuda.to_device(np.zeros(1, dtype=np.float32))
        cuda.synchronize()
        t0 = time.perf_counter()
        kernel[bpg, tpb](d_arr, d_t)
        cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    return sum(times) / len(times)

print(f"Naive time:  {time_kernel(naive_sum_kernel):.4f} ms")
print(f"Atomic time: {time_kernel(atomic_sum_kernel):.4f} ms")

NAIVE KERNEL
Trial 1: output = 2.0, lost updates = 49998.0
Trial 2: output = 2.0, lost updates = 49998.0
Trial 3: output = 2.0, lost updates = 49998.0
Trial 4: output = 2.0, lost updates = 49998.0
Trial 5: output = 2.0, lost updates = 49998.0
ATOMIC output = 50000.0
Naive time:  0.1649 ms
Atomic time: 0.3225 ms


In [14]:
import math

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)
    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)
        if intensity < 0.0:
            intensity = 0.0
        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2)
cuda.synchronize()
result_canvas = d_canvas.copy_to_host()
print(f"Generated 2D matrix shape: {result_canvas.shape}. Center value: {result_canvas[512, 512]:.3f}")
print(f"Blocks: {blocks_2d}, total = {blocks_2d[0] * blocks_2d[1]}")
print(f"Corner (0,0): {result_canvas[0, 0]:.3f}")

Generated 2D matrix shape: (1024, 1024). Center value: 1.000
Blocks: (64, 64), total = 4096
Corner (0,0): 0.000


In [16]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))
    # Include GPU name
    name = cuda.get_current_device().name
    h.update(name if isinstance(name, bytes) else name.encode('utf-8'))
    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))
    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103148"   # replace with your real ID, keep the quotes
CROSSOVER_N = 10000                   # your N* from Q1

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: F05A1886D713ECCE


'F05A1886D713ECCE'